# RAG Pipeline Demo — Pháp Luật Ma Tuý & Nghệ Sĩ Việt Nam

**Sinh viên:** Nguyễn Thành Huy — MSSV: 2A202600764  
**Chủ đề:** Pháp luật Việt Nam về ma tuý + Bài báo về nghệ sĩ liên quan ma tuý  
**Stack:** ChromaDB + BM25 + RRF + Anthropic Claude (RAG pipeline end-to-end)

---

## Kiến Trúc Pipeline

```
Query
  ├─→ Semantic Search (ChromaDB dense, paraphrase-multilingual-MiniLM-L12-v2)
  │
  ├─→ Lexical Search  (BM25Okapi, rank-bm25)
  │
  ├─→ RRF Merge       (Reciprocal Rank Fusion, k=60)
  │
  ├─→ Reranking       (RRF / MMR / Jina cross-encoder)
  │
  ├─→ Fallback        (PageIndex vectorless nếu score < threshold)
  │
  └─→ Generation      (Claude Haiku + citation + lost-in-middle reordering)
```

In [ ]:
import sys
sys.path.insert(0, '..')  # Thêm project root vào path

from pathlib import Path
import json

PROJECT_ROOT = Path('..').resolve()
print(f'Project root: {PROJECT_ROOT}')

## Task 1 & 2 — Dữ Liệu Thu Thập

In [ ]:
legal_dir = PROJECT_ROOT / 'data' / 'landing' / 'legal'
news_dir  = PROJECT_ROOT / 'data' / 'landing' / 'news'

legal_files = [f for f in legal_dir.iterdir() if f.suffix.lower() in {'.pdf', '.docx', '.doc'}]
news_files  = [f for f in news_dir.iterdir()  if f.suffix.lower() in {'.json', '.html'}]

print(f'📁 Văn bản pháp luật: {len(legal_files)} files')
for f in legal_files:
    print(f'   {f.name}  ({f.stat().st_size/1024:.1f} KB)')

print(f'\n📰 Bài báo: {len(news_files)} files')
for f in news_files:
    data = json.loads(f.read_text(encoding='utf-8'))
    print(f'   [{data["source"]}] {data["title"][:70]}')

## Task 3 — Markdown Conversion

In [ ]:
std_dir = PROJECT_ROOT / 'data' / 'standardized'
md_files = list(std_dir.rglob('*.md'))

print(f'✅ Đã convert {len(md_files)} files sang Markdown')
for f in md_files:
    content = f.read_text(encoding='utf-8')
    doc_type = 'legal' if 'legal' in str(f) else 'news'
    print(f'   [{doc_type}] {f.name}: {len(content):,} chars')

# Preview một file
print('\n--- Preview: luat-73-2021 (500 chars đầu) ---')
legal_md = list((std_dir / 'legal').rglob('*.md'))
if legal_md:
    print(legal_md[0].read_text(encoding='utf-8')[:500])

## Task 4 — Chunking & Indexing

In [ ]:
from src.task4_chunking_indexing import (
    load_documents, chunk_documents, get_chroma_collection,
    CHUNK_SIZE, CHUNK_OVERLAP, EMBEDDING_MODEL
)

print(f'⚙ Cấu hình chunking:')
print(f'   Strategy: RecursiveCharacterTextSplitter')
print(f'   chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}')
print(f'\n⚙ Embedding model: {EMBEDDING_MODEL}')
print(f'   Dimension: 384, Multilingual (hỗ trợ tiếng Việt)')

docs = load_documents()
chunks = chunk_documents(docs)
collection = get_chroma_collection()

print(f'\n📊 Thống kê:')
print(f'   Documents loaded: {len(docs)}')
print(f'   Chunks created:   {len(chunks)}')
print(f'   ChromaDB count:   {collection.count()}')

# Phân bố chunks theo loại
legal_chunks = sum(1 for c in chunks if c['metadata']['type'] == 'legal')
news_chunks  = sum(1 for c in chunks if c['metadata']['type'] == 'news')
print(f'   Legal chunks: {legal_chunks}, News chunks: {news_chunks}')

## Task 5 — Semantic Search (Dense Retrieval)

In [ ]:
from src.task5_semantic_search import semantic_search

query = 'hình phạt tội tàng trữ trái phép chất ma tuý'
print(f'🔍 Semantic Search Query: "{query}"')
print('-' * 70)

results = semantic_search(query, top_k=5)
for i, r in enumerate(results, 1):
    print(f'{i}. [score={r["score"]:.4f}] [{r["metadata"]["type"]}] {r["metadata"]["source"]}')
    print(f'   {r["content"][:120].strip()}...')
    print()

## Task 6 — Lexical Search (BM25)

In [ ]:
from src.task6_lexical_search import lexical_search, build_bm25_index

print('Building BM25 index...')
build_bm25_index()

query = 'nghệ sĩ bị bắt sử dụng ma tuý'
print(f'\n🔍 BM25 Query: "{query}"')
print('-' * 70)

results = lexical_search(query, top_k=5)
for i, r in enumerate(results, 1):
    print(f'{i}. [bm25={r["raw_bm25_score"]:.2f}|norm={r["score"]:.4f}] [{r["metadata"]["type"]}]')
    print(f'   {r["content"][:120].strip()}...')
    print()

## Task 7 — Reranking (RRF + MMR + Cross-Encoder)

In [ ]:
from src.task7_reranking import rerank, rerank_rrf, rerank_mmr

query = 'hình phạt tội tàng trữ ma tuý'

dense  = semantic_search(query, top_k=10)
sparse = lexical_search(query, top_k=10)

print('=== RRF (Reciprocal Rank Fusion) ===')
rrf_results = rerank_rrf([dense, sparse], top_k=5)
for i, r in enumerate(rrf_results, 1):
    print(f'{i}. [rrf={r["score"]:.6f}] [{r["metadata"]["type"]}] {r["content"][:90].strip()}')

print('\n=== MMR (Maximal Marginal Relevance, λ=0.7) ===')
all_candidates = dense + sparse
mmr_results = rerank_mmr([], all_candidates, top_k=5, lambda_param=0.7)
for i, r in enumerate(mmr_results, 1):
    print(f'{i}. [score={r["score"]:.4f}] [{r["metadata"]["type"]}] {r["content"][:90].strip()}')

## Task 8 — PageIndex Vectorless RAG

In [ ]:
from src.task8_pageindex_vectorless import pageindex_search

query = 'cai nghiện bắt buộc luật phòng chống ma tuý'
print(f'🔍 PageIndex (vectorless) Query: "{query}"')
print('(Dùng keyword fallback nếu không có PAGEINDEX_API_KEY)')
print('-' * 70)

results = pageindex_search(query, top_k=3)
for i, r in enumerate(results, 1):
    print(f'{i}. [score={r["score"]:.4f}] [{r["source"]}] {r["metadata"]["source"]}')
    print(f'   {r["content"][:120].strip()}...')
    print()

## Task 9 — Full Retrieval Pipeline (Hybrid + Fallback)

In [ ]:
from src.task9_retrieval_pipeline import retrieve

test_queries = [
    'Hình phạt cho tội tàng trữ trái phép chất ma tuý theo Bộ luật Hình sự',
    'Những nghệ sĩ nào đã bị bắt vì liên quan tới ma tuý',
    'Luật Phòng chống ma tuý 2021 quy định gì về cai nghiện bắt buộc',
]

for q in test_queries:
    print(f'\n🔍 Query: "{q}"')
    print('-' * 70)
    results = retrieve(q, top_k=3)
    for i, r in enumerate(results, 1):
        print(f'  {i}. [score={r["score"]:.4f}] [via={r["source"]}] [{r["metadata"]["type"]}]')
        print(f'     {r["content"][:100].strip()}...')

## Task 10 — Generation Có Citation (End-to-End RAG)

In [ ]:
from src.task10_generation import generate_with_citation, reorder_for_llm, format_context

# Demo reorder_for_llm
print('=== Demo: reorder_for_llm (Lost-in-the-Middle Prevention) ===')
dummy_chunks = [{'content': f'Chunk {i}', 'score': 1.0 - i*0.1} for i in range(5)]
reordered = reorder_for_llm(dummy_chunks)
print('Input  (score desc):', [c['content'] for c in dummy_chunks])
print('Output (reordered): ', [c['content'] for c in reordered])
print('Chunk quan trọng nhất (rank 1) vẫn ở đầu ✓')

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')

if not os.getenv('ANTHROPIC_API_KEY') or os.getenv('ANTHROPIC_API_KEY') == 'your_anthropic_api_key_here':
    print('⚠ ANTHROPIC_API_KEY chưa được set trong .env')
    print('  Thêm API key vào file .env để chạy generation')
    print('  Lấy key tại: https://console.anthropic.com/')
else:
    query = 'Hình phạt cho tội tàng trữ trái phép chất ma tuý theo pháp luật Việt Nam?'
    print(f'Q: {query}')
    print('=' * 70)
    
    result = generate_with_citation(query)
    
    print(f'A:\n{result["answer"]}')
    print(f'\n[{len(result["sources"])} nguồn | via {result["retrieval_source"]}]')
    usage = result.get('usage', {})
    print(f'[tokens: {usage.get("input_tokens",0)} in / {usage.get("output_tokens",0)} out]')

## Tổng Kết

| Task | Nội dung | Trạng thái |
|------|----------|------------|
| 1 | Thu thập 5 văn bản pháp luật (DOC) | ✅ |
| 2 | Crawl 10 bài báo (JSON + metadata) | ✅ |
| 3 | Convert sang Markdown (15 files) | ✅ |
| 4 | Chunking (800/150) + ChromaDB indexing | ✅ |
| 5 | Semantic search (cosine similarity) | ✅ |
| 6 | Lexical search (BM25Okapi) | ✅ |
| 7 | Reranking (RRF + MMR + Jina cross-encoder) | ✅ |
| 8 | PageIndex vectorless RAG (keyword fallback) | ✅ |
| 9 | Hybrid pipeline + fallback logic | ✅ |
| 10 | Generation có citation (Claude Haiku) | ✅ |

### Điểm Kỹ Thuật Nổi Bật
- **Hybrid Search**: Kết hợp dense (semantic) + sparse (BM25) → RRF merge để tận dụng cả 2 phương pháp
- **Lost-in-the-Middle**: Sắp xếp chunks theo pattern đặc biệt để LLM không quên thông tin ở giữa
- **Citation**: Mọi thông tin pháp lý đều có citation [Nguồn, Điều X/Năm]
- **Fallback**: 3 tầng fallback: hybrid → pageindex → keyword search